# 📦 [Project Name] – [Notebook Purpose]

Briefly describe the notebook’s purpose: what it models, why it's important, and what makes this analysis different (e.g., forward-looking, hybrid modelling, simulation-driven, etc.).

---

### 🎯 Objective

Describe what you're trying to predict or simulate, and why it's challenging. Consider framing the difference between prediction vs inference if relevant.

> Example: Predict [target variable] using only [input space] **before** [event/intervention].

---

### 🔍 Feature Scope

Clarify which features are **valid inputs** vs those that are **excluded or used for validation**.

#### 📌 Included Inputs
- `Feature_1`
- `Feature_2`
- `Feature_n`
- [Constants or test-based parameters]

#### 🧹 Excluded Features
- [Real-time or output-side variables]
- [Post-process or derived estimates]

---

### 🧠 Modelling Strategy

#### ⚒️ Feature Engineering Approach
- Consider interaction terms, ratios, lag effects
- Stratify by known breakpoints or metadata flags

#### 🧪 Hybrid / Physics-Guided Elements
- Use domain models or physical constraints to create a baseline
- Fit statistical/ML model on **residuals from physical model**

---

### ⚠️ Data Limitations

Acknowledge:
- Noise, gaps, or bias in the dataset
- Assumptions around unmeasured drivers

> Design for guidance, not precision — e.g., rank-based outputs, banded predictions, or risk flags.

---

### 🧭 Summary of Approach

| Step | Action                                                                                   |
|------|-------------------------------------------------------------------------------------------|
| 1️⃣   | Clean and validate feature set                                                           |
| 2️⃣   | Engineer additional inputs based on domain or structure                                  |
| 3️⃣   | Integrate physical or kinetic baselines (if relevant)                                    |
| 4️⃣   | Train and test predictive model on chosen target                                         |
| 5️⃣   | Assess decision utility (e.g. improved planning, early warning)                          |


# STANDARD CELLS

## 📂 Load Data & Initial Setup

This section loads the dataset(s), sets up basic packages and display settings, and ensures a consistent environment for downstream analysis.

Tasks:
- Import required libraries (Pandas, NumPy, Matplotlib, etc.)
- Load raw data files (e.g. CSV, Excel, Parquet)
- Parse dates and set index (if applicable)
- Configure Pandas display options and global warnings

> Ensure file paths and date formats are correct before proceeding.

## ⚙️ Define Constants & User Parameters

This section defines model constants, site-specific assumptions, and user-configurable parameters.

Includes:
- Physical constants or site-specific values (e.g. density, locked gold %, grain size)
- Configurable thresholds for model filtering or grouping
- Toggles to enable/disable specific modelling behaviours

> These parameters drive key behaviours in cleansing, feature engineering, and modelling stages.

## 🧹 Data Cleansing & Preprocessing

This section performs initial data cleaning to prepare for analysis.

Steps typically include:
- Handling missing values and interpolating where appropriate
- Flagging invalid or out-of-range records
- Converting units or renaming columns for consistency
- Filtering or trimming records (e.g. incomplete days, sensor dropouts)

> Cleansing logic should be conservative — retain as much valid data as possible while removing obvious anomalies.

## 🧰 Feature Engineering

This section derives new features to enhance model performance.

Common operations:
- Create interaction terms (e.g. Grade × FinesFraction, Throughput / PSD)
- Extract temporal or operational flags (e.g. shift, supplier batch, post-changeover)
- Apply physical models to derive baseline expectations (e.g. Lima & Hodouin CN demand)
- Categorise variables into bands or strata (e.g. High/Med/Low O₂)

> Thoughtful feature engineering often yields more benefit than swapping models.

## 📊 Define Plotting Helpers

Reusable plotting functions for visual diagnostics and model evaluation.

Includes:
- Scatter plots and residual plots
- Trend overlays and prediction vs actual charts
- Monthly or period summaries
- Heatmaps and interaction visualisations

> Use these throughout the notebook to keep visual logic consistent and concise.

# POST PLOTTING CELLS


### 📈 What the plot shows
- **X-axis:** [Short, descriptive label]
- **Y-axis:** [Short, descriptive label]
- **Trend / Shape / Highlight:** [Brief description, e.g., "Flat trend", "Nonlinear upward curve", "High variance at low CN"]

### 🧠 Interpretation
- [Start with the main insight in bold if possible]
- [Add context or counterpoints]
- [Relate back to hypotheses or prior findings]

### 🚩 Key Takeaway
> [Use a single sentence summary in quote block]
- [Support with 1–2 concise bullet points if needed]
- [Keep this section as the “memory anchor” for downstream use]

💻 STANDARD MARKDOWN CELL
📊 PLOTTING CELL
🧮 MODELLING CELL

# PLOTTING HELPERS

In [ ]:
# --- Plotting Helper Functions ---

def plot_bar_error(x, y, title, xlabel, ylabel):
	"""
	Bar plot of error or difference values.

	Parameters:
	- x: x-axis values (e.g., categories or dates)
	- y: y-axis values (e.g., % error or differences)
	- title: plot title
	- xlabel: label for x-axis
	- ylabel: label for y-axis
	"""
	plt.figure(figsize=(12, 6))
	plt.bar(x, y, color="skyblue")
	plt.axhline(0, color='red', linestyle='--', linewidth=1)  # zero reference line
	plt.title(title)
	plt.xlabel(xlabel)
	plt.ylabel(ylabel)
	plt.xticks(rotation=45)
	plt.tight_layout()
	plt.show()


def plot_line_with_error_bands(x, y1, y2, title, xlabel, ylabel,
							   y1_label="Reported", y2_label="Estimated"):
	"""
	Line plot with two series and shaded error band.

	Parameters:
	- x: x-axis values
	- y1: first line series (e.g., reported values)
	- y2: second line series (e.g., estimated values)
	- title, xlabel, ylabel: plot metadata
	- y1_label, y2_label: legend labels
	"""
	plt.figure(figsize=(12, 6))
	plt.plot(x, y1, label=y1_label, marker='o')
	plt.plot(x, y2, label=y2_label, marker='s')
	plt.fill_between(x, y1, y2, color='gray', alpha=0.3, label="Error Band")  # shaded error area
	plt.title(title)
	plt.xlabel(xlabel)
	plt.ylabel(ylabel)
	plt.xticks(rotation=45)
	plt.legend()
	plt.grid(True)
	plt.tight_layout()
	plt.show()


def plot_scatter_with_regression(x, y, title, xlabel, ylabel):
	"""
	Scatter plot with fitted regression line.

	Parameters:
	- x, y: data points
	- title, xlabel, ylabel: plot metadata
	"""
	plt.figure(figsize=(6, 5))
	plt.scatter(x, y, alpha=0.6)
	regression_line = np.poly1d(np.polyfit(x, y, 1))
	plt.plot(np.unique(x), regression_line(np.unique(x)), color='red', linewidth=2)
	plt.title(title)
	plt.xlabel(xlabel)
	plt.ylabel(ylabel)
	plt.grid(True)
	plt.tight_layout()
	plt.show()


def plot_multiple_lines(df, x_col, y_series, title="Line Plot", x_label="X", y_label="Y", colours=None):
	"""
	Plot multiple time series or variables on the same line plot.

	Parameters:
	- df: DataFrame containing data
	- x_col: column name for x-axis
	- y_series: list of dicts with keys:
		- 'col': y column name
		- 'label': label for legend
		- 'style': dict of matplotlib style options (optional)
	- title, x_label, y_label: plot metadata
	- colours: list of colours to apply to each line (optional)
	"""
	plt.figure(figsize=(12, 6))
	
	for i, series in enumerate(y_series):
		y_col = series["col"]
		label = series.get("label", y_col)
		style = series.get("style", {})
		colour = colours[i] if colours and i < len(colours) else None
		plt.plot(df[x_col], df[y_col], label=label, color=colour, **style)

	plt.title(title)
	plt.xlabel(x_label)
	plt.ylabel(y_label)
	plt.xticks(rotation=45)
	plt.grid(True)
	plt.legend()
	plt.tight_layout()
	plt.show()


def plot_actual_vs_predicted_scatter(y_actual, y_pred, title="Model Performance",
									 xlabel="Actual", ylabel="Predicted"):
	"""
	Scatter plot comparing actual vs predicted values with a y = x reference line.

	Parameters:
	- y_actual: true values
	- y_pred: predicted values
	- title, xlabel, ylabel: plot metadata
	"""
	plt.figure(figsize=(6, 5))
	plt.scatter(y_actual, y_pred, alpha=0.6)
	min_val = min(y_actual.min(), y_pred.min())
	max_val = max(y_actual.max(), y_pred.max())
	plt.plot([min_val, max_val], [min_val, max_val], 'r--')  # y = x reference
	plt.xlabel(xlabel)
	plt.ylabel(ylabel)
	plt.title(title)
	plt.grid(True)
	plt.tight_layout()
	plt.show()


def plot_residuals(y_actual, residuals, title="Residuals vs Actual",
				   xlabel="Actual", ylabel="Residual (Actual − Predicted)",
				   alpha=0.6):
	"""
	Scatter plot of residuals (error) vs actual values.

	Parameters:
	- y_actual: true values
	- residuals: actual - predicted
	- title, xlabel, ylabel: plot metadata
	- alpha: transparency for points
	"""
	plt.figure(figsize=(7, 5))
	plt.scatter(y_actual, residuals, alpha=alpha)
	plt.axhline(0, color='red', linestyle='--')  # zero residual reference
	plt.xlabel(xlabel)
	plt.ylabel(ylabel)
	plt.title(title)
	plt.grid(True)
	plt.tight_layout()
	plt.show()
	
def plot_residuals_over_time(df, date_col="Date", residual_col="Residual",
							 title="Residuals Over Time", xlabel="Date", ylabel="Residual (Actual − Predicted)"):
	"""
	Line plot of residuals over time with zero reference line.

	Parameters:
	- df: DataFrame containing date and residual columns
	- date_col: column name representing dates
	- residual_col: column name for residual values
	- title, xlabel, ylabel: plot metadata
	"""
	plt.figure(figsize=(12, 4))
	plt.plot(df[date_col], df[residual_col], marker='o', linestyle='-', alpha=0.7)
	plt.axhline(0, color='red', linestyle='--')
	plt.title(title)
	plt.xlabel(xlabel)
	plt.ylabel(ylabel)
	plt.tight_layout()
	plt.grid(True)
	plt.show()

def plot_usage_with_outliers(df, date_col="Date",
							 actual_col="Calc_NaCN_Used_Kg",
							 predicted_col="Predicted_NaCN_Used_kg",
							 outlier_col="Outlier",
							 title="NaCN Usage with Outlier Days Highlighted",
							 xlabel="Date", ylabel="NaCN Used (kg)"):
	"""
	Line plot of actual vs predicted NaCN usage with outliers highlighted.

	Parameters:
	- df: DataFrame containing time series and outlier flag
	- date_col: column with dates
	- actual_col: actual usage column
	- predicted_col: predicted usage column
	- outlier_col: boolean column indicating outliers
	- title, xlabel, ylabel: plot metadata
	"""
	plt.figure(figsize=(14, 6))
	
	# Plot actual and predicted usage
	plt.plot(df[date_col], df[actual_col], label="Actual", color="black", linewidth=1.5)
	plt.plot(df[date_col], df[predicted_col], label="Predicted", color="green", linewidth=1.5)
	
	# Highlight outliers
	plt.scatter(df.loc[df[outlier_col], date_col],
				df.loc[df[outlier_col], actual_col],
				color="red", label="Outliers", zorder=5)
	
	plt.xlabel(xlabel)
	plt.ylabel(ylabel)
	plt.title(title)
	plt.legend()
	plt.grid(True)
	plt.tight_layout()
	plt.show()

def plot_grouped_dual_line_comparison(
	df: pd.DataFrame,
	group_by: str,
	generate_lines_fn: Callable[[pd.DataFrame], Tuple[List[str], dict]],
	title_fn: Callable[[str], str] = lambda g: str(g),
	ylabel: str = "Value",
	xlabel: str = "X-Axis",
	legend_labels: List[str] = None,
	ncols: int = 3,
	figsize: Tuple[int, int] = (18, 4),
):
	"""
	Generic plotting function for grouped line comparisons.

	Parameters:
		df (pd.DataFrame): Source dataframe
		group_by (str): Column name to group by (e.g. 'Month')
		generate_lines_fn (Callable): Function that returns (x_labels, y_lines_dict)
		title_fn (Callable): Function to convert group key into subplot title
		ylabel (str): Label for y-axis
		xlabel (str): Label for x-axis
		legend_labels (List[str]): Custom legend labels (overrides dict keys if given)
		ncols (int): Number of columns in subplot grid
		figsize (Tuple): Base (width, height) per row of the subplot grid
	"""
	df[group_by] = pd.to_datetime(df["Date"]).dt.to_period("M")
	grouped = df.groupby(group_by)

	n_groups = len(grouped)
	nrows = int(np.ceil(n_groups / ncols))
	fig, axes = plt.subplots(nrows, ncols, figsize=(figsize[0], figsize[1] * nrows), sharey=True)
	axes = axes.flatten()

	for i, (group_key, group_df) in enumerate(grouped):
		x_labels, lines_dict = generate_lines_fn(group_df)

		ax = axes[i]
		for j, (label, y_values) in enumerate(lines_dict.items()):
			display_label = legend_labels[j] if legend_labels and j < len(legend_labels) else label
			colour = def_colours[j % len(def_colours)]
			ax.plot(x_labels, y_values, label=display_label, marker='o', color=colour)

		ax.set_title(title_fn(group_key))
		ax.set_xlabel(xlabel)
		ax.set_ylabel(ylabel)
		ax.set_xticks(range(len(x_labels)))
		ax.set_xticklabels(x_labels, rotation=45)
		ax.grid(True)
		ax.legend()

	for j in range(i + 1, len(axes)):
		fig.delaxes(axes[j])

	plt.tight_layout()
	plt.show()


# EMOJIS

In [ ]:
EmojiDict = {
    # Numbers
    "numbers": "🔢",
    "zero": "0️⃣",
    "one": "1️⃣",
    "two": "2️⃣",
    "three": "3️⃣",
    "four": "4️⃣",
    "five": "5️⃣",
    "six": "6️⃣",
    "seven": "7️⃣",
    "eight": "8️⃣",
    "nine": "9️⃣",
    "ten": "🔟",
    "eleven": "1️⃣1️⃣",
    "twelve": "1️⃣2️⃣",
    "thirteen": "1️⃣3️⃣",
    "fourteen": "1️⃣4️⃣",
    "fifteen": "1️⃣5️⃣",
    "sixteen": "1️⃣6️⃣",
    "seventeen": "1️⃣7️⃣",
    "eighteen": "1️⃣8️⃣",
    "nineteen": "1️⃣9️⃣",
    "twenty": "2️⃣0️⃣",
    "twenty_one": "2️⃣1️⃣",
    "twenty_two": "2️⃣2️⃣",
    "twenty_three": "2️⃣3️⃣",
    "twenty_four": "2️⃣4️⃣",
    "twenty_five": "2️⃣5️⃣",
    "twenty_six": "2️⃣6️⃣",
    "twenty_seven": "2️⃣7️⃣",
    "twenty_eight": "2️⃣8️⃣",
    "twenty_nine": "2️⃣9️⃣",
    "thirty": "3️⃣0️⃣",

    # Work, Tools, Building
    "tools": "🛠️",
    "gear": "⚙️",
    "wrench": "🔧",
    "hammer": "🔨",
    "saw": "🪚",
    "axe": "🪓",
    "brick": "🧱",
    "factory": "🏭",
    "construction_site": "🏗️",
    "magnet": "🧲",
    "ladder": "🪜",
    
    # Data, Documents
    "document": "📄",
    "folder": "📂",
    "package": "📦",
    "clipboard": "📋",
    "calendar": "📅",
    "chart_up": "📈",
    "chart_down": "📉",
    "bar_chart": "📊",
    "pushpin": "📌",
    "notebook": "📓",
    "bookmark": "🔖",
    "receipt": "🧾",
    "file_cabinet": "🗄️",
    "page_with_curl": "📃",
    
    # Launch, Progress, Movement
    "rocket": "🚀",
    "target": "🎯",
    "plane_takeoff": "🛫",
    "plane_landing": "🛬",
    "runner": "🏃‍♂️",
    "cyclist": "🚴‍♂️",
    "skier": "⛷️",
    "climber": "🧗",
    "summit": "🏔️",
    "parachute": "🪂",
    "ship": "🚢",
    "car": "🚗",
    "motorbike": "🏍️",
    "truck": "🚚",
    
    # Errors, Warnings
    "error": "❌",
    "stop": "🛑",
    "warning": "⚠️",
    "bomb": "💣",
    "fire": "🔥",
    "explosion": "💥",
    "siren": "🚨",
    "cross_mark": "❌",
    "question_mark": "❓",
    "exclamation_mark": "❗",
    

    # Cycles, Arrows, Sync
    "refresh": "🔄",
    "repeat": "🔁",
    "shuffle": "🔀",
    "recycle": "♻️",
    "back_arrow": "⬅️",
    "forward_arrow": "➡️",
    "up_arrow": "⬆️",
    "down_arrow": "⬇️",
    "clockwise": "🔄",

    # Communication
    "megaphone": "📢",
    "loudspeaker": "📣",
    "email": "📧",
    "inbox": "📥",
    "outbox": "📤",
    "telephone": "📞",
    "chat": "💬",
    "speech_balloon": "💬",
    "thinking": "🤔",
    
    # Achievement, Celebration
    "trophy": "🏆",
    "gold_medal": "🥇",
    "silver_medal": "🥈",
    "bronze_medal": "🥉",
    "star": "⭐",
    "sparkle": "✨",
    "ribbon": "🎀",
    "confetti_ball": "🎊",
    "fireworks": "🎆",

    # Location, Mapping
    "earth": "🌍",
    "mountain": "⛰️",
    "tent": "⛺",
    "beach": "🏖️",
    "desert": "🏜️",
    "snow_capped_mountain": "🏔️",
    "compass": "🧭",
    "map": "🗺️",
    "anchor": "⚓",

    # Finance, Money
    "money_bag": "💰",
    "coin": "🪙",
    "dollar": "💵",
    "yen": "💴",
    "euro": "💶",
    "pound": "💷",
    "credit_card": "💳",
    "safe": "🧰",
    "wallet": "👛",
    "stock_up": "📈",
    "stock_down": "📉",
    
    # Safety, PPE
    "shield": "🛡️",
    "safety_vest": "🦺",
    "boots": "🥾",
    "helmet_man": "👷‍♂️",
    "helmet_woman": "👷‍♀️",
    "emergency_exit": "🚪",
    
    # Science, Research
    "test_tube": "🧪",
    "microscope": "🔬",
    "dna": "🧬",
    "petri_dish": "🧫",
    "satellite": "🛰️",
    "telescope": "🔭",
    
    # Devices, Computers
    "computer": "💻",
    "laptop": "💻",
    "printer": "🖨️",
    "mouse": "🖱️",
    "mobile_phone": "📱",
    "signal_strength": "📶",
    "battery": "🔋",
    "satellite_antenna": "📡",
    "joystick": "🕹️",

    # Cleaning
    "broom": "🧹",
    "bucket": "🪣",
    "sponge": "🧽",
    "soap": "🧼",
    "vacuum": "🧹",
    
    # Fun
    "video_game": "🎮",
    "joystick_fun": "🕹️",
    "dice": "🎲",
    "puzzle_piece": "🧩",
    "balloon": "🎈",
    "party_popper": "🎉",
    
    # People and Teams
    "person": "👤",
    "team": "👥",
    "handshake": "🤝",
    "boss": "👨‍💼",
    "engineer": "👷‍♂️",
    "astronaut": "👨‍🚀",
    "scientist": "🧑‍🔬",
    "worker": "👷",

    # Weather and Nature
    "sun": "☀️",
    "rain": "🌧️",
    "thunderstorm": "⛈️",
    "tornado": "🌪️",
    "snowflake": "❄️",
    "rainbow": "🌈",
    "volcano": "🌋",
    "tree": "🌳",
    "cactus": "🌵",

    # Other
    "coloured_panels": "📚"

}